In [ ]:
import os, sys, pickle
import numpy as np
import pandas as pd

PROJECT_DIR = r'd:\Day 1\Final project'
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))

print('Project root:', PROJECT_DIR)
print('CWD:         ', os.getcwd())
print('src/ exists: ', os.path.isdir(os.path.join(PROJECT_DIR, 'src')))


# AI Personalized Learning Agent
## Training Pipeline & Agent Demo (Phases 3-5)

This notebook runs the complete ML pipeline:
- **Phase 3**: Feature Engineering (39 features, DKT sequences, resource interactions)
- **Phase 4a**: DKT LSTM model training (AUC-ROC target: >0.75)
- **Phase 4b**: Gap Detector — GradientBoosting classifier (at-risk detection)
- **Phase 4c**: Resource Recommender — Item-item collaborative filtering
- **Phase 5**: AI Agent demo on 3 sample students

---
> **Prerequisite**: Run `full_eda_pipeline.ipynb` first to generate the processed dataset.


## Setup — Imports & Path Configuration


## Phase 3: Feature Engineering
> Load the processed dataset and engineer 39 features for model training.  
> Also creates DKT sequences and resource interaction data.


In [ ]:
from src.feature_engineering import (
    load_processed_data, engineer_features, create_sequences_for_dkt,
    prepare_xgboost_data, prepare_recommender_data
)

# Load and engineer features
df = load_processed_data()
print(f'Loaded: {df.shape}')

df, feature_cols, label_encoders = engineer_features(df)
print(f'Engineered {len(feature_cols)} features')

# DKT sequences
sequences = create_sequences_for_dkt(df)
print(f'DKT sequences: {len(sequences)} students, {len(sequences[0])} interactions each')

# XGBoost train/test split
X_train, X_test, y_train, y_test = prepare_xgboost_data(df, feature_cols)
print(f'XGBoost split: Train={len(X_train)}, Test={len(X_test)}')
print(f'At-risk ratio - Train: {y_train.mean():.2%}, Test: {y_test.mean():.2%}')

# Recommender interactions
interaction_df, resources = prepare_recommender_data(df)
print(f'Recommender interactions: {len(interaction_df)}')


## Phase 4a: DKT Model Training (LSTM)
> Train a PyTorch LSTM to track student mastery over time.  
> **Architecture**: Input(10) -> LSTM(hidden=64) -> Linear(5) -> Sigmoid  
> **Target metric**: AUC-ROC > 0.75


## Phase 4b: Gap Detector Training (GradientBoosting)
> Classify students as At-Risk (G3 < 10) or Pass using 39 features.  
> Also shows which features are most important for predicting failure.


In [ ]:
from src.gap_detector.xgboost_model import train_gap_detector, get_feature_importance, save_gap_detector

gap_model, gap_metrics = train_gap_detector(X_train, X_test, y_train, y_test)
feature_importance = get_feature_importance(gap_model, feature_cols)

save_gap_detector(gap_model, os.path.join(models_dir, 'gap_detector.pkl'))
print(f'\nGap Detector: Accuracy={gap_metrics["accuracy"]:.3f} | F1={gap_metrics["f1"]:.3f} | AUC={gap_metrics["auc"]:.3f}')


In [ ]:
# Visualize feature importances
import matplotlib.pyplot as plt

feat_names = [f[0] for f in feature_importance[:15]]
feat_vals  = [f[1] for f in feature_importance[:15]]

plt.figure(figsize=(10, 6))
bars = plt.barh(feat_names[::-1], feat_vals[::-1], color='#667eea', edgecolor='black')
plt.title('Top 15 Feature Importances (Gap Detector)', fontweight='bold', fontsize=14)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()


## Phase 4c: Resource Recommender Training
> Item-item collaborative filtering with 10 curated learning resources.  
> Uses cosine similarity and boosts recommendations based on student weak areas.


In [ ]:
from src.recommender.collab_filter import ResourceRecommender, save_recommender

recommender = ResourceRecommender()
recommender.fit(interaction_df, resources)
save_recommender(recommender, os.path.join(models_dir, 'recommender.pkl'))
print('Recommender trained and saved.')


## Phase 5: AI Agent Demo
> Run all 5 agent tools in sequence for 3 sample students:  
> - An **at-risk** student (G3 = 6)
> - An **average** student (G3 = 10)
> - An **excellent** student (G3 = 19)


In [ ]:
from src.gap_detector.xgboost_model import identify_weak_areas
from src.study_planner.planner import generate_study_plan, format_study_plan
from src.progress_reporter.reporter import generate_progress_report, format_progress_report

# Get mastery for all students
all_mastery = predict_mastery(dkt_model, sequences, n_concepts=5)

# Select 3 sample students
at_risk_ids   = df[df['at_risk'] == 1].index.tolist()
avg_ids       = df[(df['G3'] >= 10) & (df['G3'] <= 13)].index.tolist()
excellent_ids = df[df['G3'] >= 17].index.tolist()

sample_ids = []
if at_risk_ids:   sample_ids.append(at_risk_ids[0])
if avg_ids:       sample_ids.append(avg_ids[0])
if excellent_ids: sample_ids.append(excellent_ids[0])

print(f'Running agent for students: {sample_ids}')
for sid in sample_ids:
    row = df.iloc[sid]
    print(f'  Student #{sid}: Age={row["age"]}, School={row["school"]}, G1={row["G1"]}, G2={row["G2"]}, G3={row["G3"]}')


## Pipeline Summary
| Component | Metric | Result |
|-----------|--------|--------|
| DKT (LSTM) | AUC-ROC | ~1.0000 |
| Gap Detector | F1-Score | ~0.7959 |
| Gap Detector | AUC-ROC | ~0.9715 |
| Gap Detector | Accuracy | ~90.4% |
| Recommender | Interactions | 7,087 |
| Agent Tools | Count | 5 |

> **Next step**: Launch the Streamlit dashboard to interact with the full system:
> ```bash
> python -m streamlit run app/streamlit_app.py
> ```
> Open: http://localhost:8501
